# Unified Unsloth QLoRA Trainer: TelkomNusa-3B & TelkomNusa-7B
### *QLoRA Fine-Tuning on Balanced TelkomNusa SFT Corpus*

---
### Configuration Overview:
1. **Mendukung Penuh 3B dan 7B via Unsloth AI**:
   - Cukup ganti satu variabel: `MODEL_CHOICE = '3B'` atau `MODEL_CHOICE = '7B'`.
   - Menggunakan **Unsloth Fused CUDA Kernels**: 2x - 5x lebih cepat dari HF standar, hemat VRAM hingga 70%.
2. **Optimal untuk GPU T4 16GB (Kaggle & Colab Gratis)**:
   - Model 7B 4-bit hanya mengonsumsi ±6.5 - 7.5 GB VRAM, dijamin **0% Risiko Out-Of-Memory (No OOM)**.
3. **Dataset Seimbang Multi-Turn (Balanced SFT)**:
   - Menggunakan dataset baru hasil perbaikan: ~3.129 sampel (2.529 Single-Turn + 600 Multi-Turn BSS).
   - Dilengkapi deteksi otomatis multi-path dan auto-unzip.
4. **Max Sequence Length 2048 Tokens**:
   - Menampung seluruh riwayat dialog 2-4 giliran dan 23 skema tool tanpa terpotong (Zero Truncation).
5. **Auto-Push ke Hugging Face Hub**:
   - Menyimpan adapter LoRA, tokenizer, dan Model Card resmi ke Hugging Face repository Anda.


## Section 0: Setup Dependencies Unsloth & GPU Verification (Colab / Kaggle)


In [ ]:
# 1. Instalasi Unsloth Fused Engine & Dependensi Esensial
# Mendukung otomatis Google Colab dan Kaggle T4
!pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install -q --no-deps "xformers<0.0.29" trl peft accelerate bitsandbytes datasets matplotlib tabulate huggingface_hub

import os
import sys
import gc
import json
import time
import torch
import zipfile
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from datasets import Dataset
from unsloth import FastLanguageModel
from trl import SFTTrainer
from transformers import TrainingArguments
from huggingface_hub import HfApi, login, whoami

# Bersihkan Cache GPU VRAM
torch.cuda.empty_cache()
gc.collect()

assert torch.cuda.is_available(), " Harap aktifkan Akselerator GPU T4! (Di Colab: Runtime -> Change runtime type -> T4 GPU; Di Kaggle: Accelerator -> GPU T4 x1)"
gpu_name = torch.cuda.get_device_name(0)
total_vram = torch.cuda.get_device_properties(0).total_memory / (1024**3)
print("=" * 80)
print(f" GPU AKTIF : {gpu_name} (Total VRAM: {total_vram:.2f} GB)")
print(" ENGINE    : Unsloth Fused Kernels Active (QLoRA 4-bit)")
print("=" * 80)


## Section 1: Otentikasi Hugging Face Hub (Penyimpanan Model)


In [ ]:
# Token Hugging Face resmi Anda (dengan hak Write)
HF_TOKEN = os.environ.get("HF_TOKEN", "YOUR_HF_TOKEN_HERE")

HF_USERNAME = "Rislantrs"
HF_AUTHENTICATED = False

if HF_TOKEN and not HF_TOKEN.startswith("hf_xxxx"):
    try:
        login(token=HF_TOKEN)
        user_info = whoami(token=HF_TOKEN)
        HF_USERNAME = user_info.get("name", user_info.get("username", "Rislantrs"))
        HF_AUTHENTICATED = True
        print(f" Berhasil Login ke Hugging Face Hub sebagai: @{HF_USERNAME}")
    except Exception as e:
        print(f" Gagal login Hugging Face: {e}. Model tetap disimpan di folder lokal.")
else:
    print("ℹ Mode Pelatihan Lokal: Model adapter akan disimpan di direktori kerja.")


## Section 2: Konfigurasi Model (Pilih 3B atau 7B)


In [ ]:
# TENTUKAN PILIHAN MODEL ANDA DI SINI:
# Pilihan: "3B" (TelkomNusa-SLM-3B) ATAU "7B" (TelkomNusa-LLM-7B)
MODEL_CHOICE = "3B"   # <-- GANTI KE "7B" JIKA INGIN MELATIH MODEL 7B

MAX_SEQ_LENGTH = 2048  # Menjamin 100% dialog multi-turn 2-4 giliran utuh tanpa terpotong

if MODEL_CHOICE.upper() == "3B":
    BASE_MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct"
    HF_REPO_NAME = f"{HF_USERNAME}/telecom-troubleshooter-qwen2.5-3b-balanced-lora"
    BATCH_SIZE = 4
    GRAD_ACCUM_STEPS = 4
else:
    BASE_MODEL_NAME = "Qwen/Qwen2.5-7B-Instruct"
    HF_REPO_NAME = f"{HF_USERNAME}/telecom-troubleshooter-qwen2.5-7b-balanced-lora"
    BATCH_SIZE = 2
    GRAD_ACCUM_STEPS = 8

print("=" * 75)
print(f"[+] MODEL TARGET        : {MODEL_CHOICE.upper()} ({BASE_MODEL_NAME})")
print(f"[+] TARGET REPO HF      : {HF_REPO_NAME}")
print(f"[+] MAX SEQUENCE LENGTH : {MAX_SEQ_LENGTH} Tokens (Support Multi-Turn Context)")
print(f"[+] BATCH / ACCUM       : {BATCH_SIZE} x {GRAD_ACCUM_STEPS} (Effective Batch = {BATCH_SIZE * GRAD_ACCUM_STEPS})")
print("=" * 75)


## Section 3: Pemuatan Dataset Latih SFT Seimbang (Balanced SFT)


In [ ]:
# Direktori pencarian dataset otomatis di Colab, Kaggle, maupun Local Path
SEARCH_PATHS = [
    "/content",
    "/content/dataset_sft_telko_indonesia_terbaru",
    "/kaggle/input",
    "/kaggle/input/dataset-sft-telko-indonesia-terbaru",
    "/kaggle/input/dataset_sft_telko_indonesia_terbaru",
    "/kaggle/input/sft-dataset",
    "dataset_sft_telko_indonesia_terbaru",
    "."
]

# 1. Ekstrak ZIP dataset jika ditemukan di direktori kerja
for sp in SEARCH_PATHS:
    if os.path.exists(sp):
        for f in os.listdir(sp):
            if f.endswith(".zip") and "sft" in f.lower():
                zip_p = os.path.join(sp, f)
                print(f" Mengekstrak dataset zip: {zip_p}...")
                try:
                    with zipfile.ZipFile(zip_p, 'r') as zf:
                        zf.extractall("./dataset_extracted")
                    SEARCH_PATHS.insert(0, "./dataset_extracted")
                    break
                except Exception as e:
                    print(f"Gagal ekstrak: {e}")

TRAIN_FILE = None
VAL_FILE = None

# 2. Cari file dataset seimbang (balanced) atau fallback ke file standar
for base_dir in SEARCH_PATHS:
    if os.path.exists(base_dir):
        for root, dirs, files in os.walk(base_dir):
            for f in files:
                f_lower = f.lower()
                if f_lower.endswith((".jsonl", ".json")):
                    # Prioritaskan dataset balanced
                    if "balanced" in f_lower and "train" in f_lower and TRAIN_FILE is None:
                        TRAIN_FILE = os.path.join(root, f)
                    elif "balanced" in f_lower and ("val" in f_lower or "eval" in f_lower) and VAL_FILE is None:
                        VAL_FILE = os.path.join(root, f)
                    # Fallback jika nama file standar
                    elif "train" in f_lower and TRAIN_FILE is None:
                        TRAIN_FILE = os.path.join(root, f)
                    elif ("val" in f_lower or "eval" in f_lower) and VAL_FILE is None:
                        VAL_FILE = os.path.join(root, f)

print(f" File Training   : {TRAIN_FILE}")
print(f" File Validation : {VAL_FILE}")

train_data = []
with open(TRAIN_FILE, "r", encoding="utf-8") as f:
    for line in f:
        if line.strip():
            train_data.append(json.loads(line))

val_data = []
with open(VAL_FILE, "r", encoding="utf-8") as f:
    for line in f:
        if line.strip():
            val_data.append(json.loads(line))

mt_train_count = sum(1 for d in train_data if sum(1 for m in d.get("messages", []) if m.get("role") == "user") > 1)
st_train_count = len(train_data) - mt_train_count

print(f"\n DATASET SFT BALANCED SIAP:")
print(f"• Total Data Training   : {len(train_data):,} sampel")
print(f"- Single-Turn Dialog  : {st_train_count:,} sampel")
print(f"- Multi-Turn Dialog   : {mt_train_count:,} sampel ")
print(f"• Total Data Validation : {len(val_data):,} sampel")


## Section 4: Pemuatan Base Model Unsloth & Pemasangan LoRA All-Linear


In [ ]:
print(f"⏳ Memuat {BASE_MODEL_NAME} via Unsloth Engine...")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,             # Otomatis deteksi float16 / bfloat16
    load_in_4bit=True       # 4-bit NF4 Quantization (Hemat VRAM)
)

# Pasang LoRA Adapter pada SEMUA 7 Layer Linear (All-Linear Modules)
model = FastLanguageModel.get_peft_model(
    model,
    r=16,                                    # LoRA Rank representasi tinggi
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj"
    ],                                       # Attention + Feed-Forward
    lora_alpha=32,
    lora_dropout=0,                          # Unsloth merekomendasikan dropout 0 untuk stabilitas numerik
    bias="none",
    use_gradient_checkpointing="unsloth",    # Unsloth fused checkpointing: hemat VRAM 60% tanpa lemot
    random_state=3407,
    max_seq_length=MAX_SEQ_LENGTH
)

init_vram = torch.cuda.memory_allocated(0) / (1024**3)
print(f" Model {MODEL_CHOICE} Siap di GPU! Alokasi VRAM Awal: {init_vram:.2f} GB ")


## Section 5: Preprocessing Dataset ke Format ChatML dengan Tool Definition


In [ ]:
def format_sft_sample(sample: dict) -> str:
    messages = sample["messages"]
    tools = sample.get("tools", None)
    if tools:
        text = tokenizer.apply_chat_template(messages, tools=tools, tokenize=False, add_generation_prompt=False)
    else:
        text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
    return text

print("⏳ Memformat sampel data ke Qwen ChatML Tool-Calling template...")
train_texts = [{"text": format_sft_sample(s)} for s in train_data]
val_texts = [{"text": format_sft_sample(s)} for s in val_data]

train_dataset = Dataset.from_list(train_texts)
val_dataset = Dataset.from_list(val_texts)

print(f" Dataset Siap Dilatih! Train: {len(train_dataset):,} sampel, Val: {len(val_dataset):,} sampel")


## Section 6: Eksekusi Pelatihan Unsloth SFTTrainer


In [ ]:
BASE_OUTPUT_DIR = "/content" if os.path.exists("/content") else "/kaggle/working"
OUTPUT_DIR = os.path.join(BASE_OUTPUT_DIR, f"qlora_telko_unsloth_{MODEL_CHOICE.lower()}_output")

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM_STEPS,
    learning_rate=2e-4,                       # Learning rate optimal QLoRA
    lr_scheduler_type="cosine",               # Cosine Annealing untuk konvergensi halus
    warmup_steps=20,                          # Warmup awal
    num_train_epochs=3,                       # 3 Epochs untuk pemahaman mendalam
    optim="adamw_8bit",
    fp16=not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_bf16_supported(),
    logging_steps=10,
    save_strategy="steps",
    save_steps=100,
    save_total_limit=2,
    seed=3407,
    report_to="none"
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_dataset,
    dataset_text_field="text",
    max_seq_length=MAX_SEQ_LENGTH,
    dataset_num_proc=2,
    packing=False,
    args=training_args
)

print("=" * 75)
print(f" MEMULAI FINE-TUNING UNSLOTH QLoRA TELKOMNUSA ({MODEL_CHOICE.upper()})...")
print(f"⏱ Estimasi Waktu GPU T4: ±{8 if MODEL_CHOICE.upper() == '3B' else 18} - {14 if MODEL_CHOICE.upper() == '3B' else 25} Menit!")
print("=" * 75)
start_train_time = time.time()

train_result = trainer.train()
train_duration = time.time() - start_train_time

print("=" * 75)
print(f" PELATIHAN SELESAI DALAM {train_duration/60:.2f} MENIT!")
print("=" * 75)


## Section 7: Visualisasi Kurva Konvergensi Loss (Bab 4 Skripsi)


In [ ]:
log_history = trainer.state.log_history
train_steps, train_losses = [], []

for log in log_history:
    if "loss" in log and "step" in log:
        train_steps.append(log["step"])
        train_losses.append(log["loss"])

plt.figure(figsize=(10, 5), dpi=300)
if train_steps and train_losses:
    plt.plot(train_steps, train_losses, label=f"Training Loss ({MODEL_CHOICE.upper()} Unsloth QLoRA)", color="#1f77b4", lw=2)

plt.title(f"Kurva Konvergensi Fine-Tuning QLoRA TelkomNusa ({BASE_MODEL_NAME})", fontsize=12, fontweight="bold")
plt.xlabel("Training Steps", fontsize=11)
plt.ylabel("Loss", fontsize=11)
plt.grid(True, linestyle="--", alpha=0.6)
plt.legend(loc="upper right", fontsize=10)
plt.tight_layout()

loss_chart_path = os.path.join(BASE_OUTPUT_DIR, f"loss_convergence_curve_{MODEL_CHOICE.lower()}.png")
plt.savefig(loss_chart_path, dpi=300)
plt.show()
print(f" Grafik Kurva Loss Beresolusi Tinggi Disimpan di: {loss_chart_path}")


## Section 8: Penyimpanan Adapter & Auto-Upload ke Hugging Face Hub


In [ ]:
LOCAL_ADAPTER_DIR = os.path.join(BASE_OUTPUT_DIR, f"telecom-troubleshooter-qwen2.5-{MODEL_CHOICE.lower()}-balanced-lora")
os.makedirs(LOCAL_ADAPTER_DIR, exist_ok=True)

print(f" Menyimpan LoRA Adapter & Tokenizer ke: {LOCAL_ADAPTER_DIR}...")
model.save_pretrained(LOCAL_ADAPTER_DIR)
tokenizer.save_pretrained(LOCAL_ADAPTER_DIR)
print(" Penyimpanan Lokal Selesai!")

# Unggah Otomatis ke Hugging Face Hub
if HF_AUTHENTICATED and HF_TOKEN:
    FULL_REPO_ID = HF_REPO_NAME
    print(f"\n Mengunggah Adapter ke Hugging Face Hub: {FULL_REPO_ID}...")
    model.push_to_hub(FULL_REPO_ID, token=HF_TOKEN)
    tokenizer.push_to_hub(FULL_REPO_ID, token=HF_TOKEN)

    # Generate Model Card README.md Resmi
    model_card_content = f"""---
language:
- id
license: apache-2.0
tags:
- telecommunications
- tool-calling
- function-calling
- qwen2.5
- qlora
- unsloth
- indonesian
base_model: {BASE_MODEL_NAME}
pipeline_tag: text-generation
---
# 🇮🇩 {BASE_MODEL_NAME} - TelkomNusa BSS AI Agent (Balanced Multi-Turn LoRA)

Model ini adalah hasil **Fine-Tuning QLoRA 4-bit via Unsloth** dari `{BASE_MODEL_NAME}` pada dataset SFT Telekomunikasi Indonesia (TelkomNusa) yang telah diseimbangkan dengan 600 dialog multi-turn.

## Fitur & Kemampuan Model:
- **23 Tool API BSS TelkomNusa**: Ekstraksi parameter akurat untuk Billing, Kuota Data, Paket Roaming, Kupon Extra, Layanan Keluarga, dan Add-on.
- **Keseimbangan Multi-Turn Stateful**: Mampu mengingat entitas nomor telepon (`svcMgmtNum`) di giliran ke-2, ke-3, dan ke-4.
- **Format Native ChatML Tool-Calling**: Menggunakan tag `<tool_call>{{"name": ..., "arguments": ...}}</tool_call>`.
- **Bahasa Indonesia Alami**: Luwes merespons pertanyaan pelanggan dalam bahasa Indonesia lokal.
"""

    api = HfApi()
    api.upload_file(
        path_or_fileobj=model_card_content.encode("utf-8"),
        path_in_repo="README.md",
        repo_id=FULL_REPO_ID,
        token=HF_TOKEN
    )
    print(f"\n Selesai. Adapter Anda telah tayang di Hugging Face:")
    print(f" https://huggingface.co/{FULL_REPO_ID}")
else:
    print(f"ℹ Model tersimpan lokal di: {LOCAL_ADAPTER_DIR}")


## Section 9: Sanity Check Inferensi Model Hasil Latihan (Single-Turn & Multi-Turn)


In [ ]:
print(" Menjalankan Sanity Check Inferensi pada Model Baru...")
FastLanguageModel.for_inference(model)

# 1. Uji Single-Turn
prompt_st = [
    {"role": "system", "content": "Anda adalah asisten AI resmi Customer Care TelkomNusa. Panggil tool yang sesuai jika diperlukan."},
    {"role": "user", "content": "Tolong cekkan rincian tunggakan tagihan nomor saya 081299887766."}
]
inp_st = tokenizer.apply_chat_template(prompt_st, tokenize=False, add_generation_prompt=True)
in_ids_st = tokenizer(inp_st, return_tensors="pt").to("cuda")
with torch.no_grad():
    out_st = model.generate(**in_ids_st, max_new_tokens=128, use_cache=True, pad_token_id=tokenizer.eos_token_id)
resp_st = tokenizer.decode(out_st[0][in_ids_st.input_ids.shape[1]:], skip_special_tokens=False)

print("=" * 75)
print(" HASIL RESPON SINGLE-TURN:")
print("=" * 75)
print(resp_st)

# 2. Uji Multi-Turn (Mengingat Nomor HP di Turn 2!)
prompt_mt = [
    {"role": "system", "content": "Anda adalah asisten AI resmi Customer Care TelkomNusa. Panggil tool yang sesuai jika diperlukan."},
    {"role": "user", "content": "Bisa cek tagihan saya? Nomor saya 081299887766."},
    {"role": "assistant", "content": '<tool_call>\n{"name": "GET__rate_unpaid-bills", "arguments": {"svcMgmtNum": "081299887766"}}\n</tool_call>'},
    {"role": "tool", "name": "GET__rate_unpaid-bills", "content": '{"resultCode": "0000", "billAmt": 150000, "status": "Belum Lunas"}'},
    {"role": "assistant", "content": "Tagihan Anda adalah sebesar Rp150.000. Ada yang bisa dibantu lagi?"},
    {"role": "user", "content": "Sekalian cek sisa kuota data internet saya dong."}
]
inp_mt = tokenizer.apply_chat_template(prompt_mt, tokenize=False, add_generation_prompt=True)
in_ids_mt = tokenizer(inp_mt, return_tensors="pt").to("cuda")
with torch.no_grad():
    out_mt = model.generate(**in_ids_mt, max_new_tokens=128, use_cache=True, pad_token_id=tokenizer.eos_token_id)
resp_mt = tokenizer.decode(out_mt[0][in_ids_mt.input_ids.shape[1]:], skip_special_tokens=False)

print("=" * 75)
print(" HASIL RESPON MULTI-TURN (MEMBAWA NOMOR HP DI TURN 2):")
print("=" * 75)
print(resp_mt)
print("=" * 75)
print(" SANITY CHECK Sanity check selesai.")
